# Example code for training CFBCT

In [ ]:
import os
import sys
sys.argv = ['run.py']
from utils.options import process_args
from dataset.dataset_survival_tcga import Generic_MIL_Survival_Dataset
from utils.utils import *

## 1. Parameters

In [43]:
args = process_args()


# Dataset: BLCA; BRCA; LUAD; UCEC; LGG; COADREAD; HNSC; STAD; 
DATASET="COADREAD"  
# Model
args.model_type='cfbct' 
# Modality: omic; path; cluster; coattn
args.mode='coattn'
# Omega_k: 0.4,0.6,0.8,1.0
args.W_k=1.0 
# Use tensorboard ? default: False
args.log_data=False
# Use function groups or pathway groups? default: True
args.apply_sig=True
# Save model? 
args.save_pkl=False
# Save model checkpoints? 
args.save_ckp=False
# 
args.data_root_dir=f""
args.split_dir=f"tcga_{DATASET.lower()}"
args.n_classes = 4
args.dataset_path='dataset_csv'

# choice fold
fold= 0
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_torch(args.seed)


## 2. Load Dataset

In [46]:
csv_path = './%s/%s_all_clean.csv.zip' % (args.dataset_path, args.split_dir)
# loading dataset 
dataset = Generic_MIL_Survival_Dataset(csv_path=csv_path,
                                        mode=args.mode,
                                        apply_sig=args.apply_sig,
                                        data_dir=args.data_root_dir,
                                        shuffle=False,
                                        seed=args.seed,
                                        print_info=True,
                                        patient_strat=False,
                                        n_bins=4,
                                        label_col='survival_months',
                                        ignore=[])

from utils.generate_utils import *
from utils.utils import get_split_loader

# split dataset 
split_dir = os.path.join('./splits', '5foldcv', args.split_dir)
train_dataset, val_dataset =dataset.return_splits(from_id=False,csv_path='{}/splits_{}.csv'.format(split_dir, fold))
# generate omic_sizes 
args.omic_sizes = train_dataset.omic_sizes
# generate loader 
train_loader = get_split_loader(train_dataset, testing = False, mode=args.mode, batch_size=args.batch_size)
val_loader = get_split_loader(val_dataset,  testing = False, mode=args.mode, batch_size=args.batch_size)

(0, 0) : 0
(0, 1) : 1
(1, 0) : 2
(1, 1) : 3
(2, 0) : 4
(2, 1) : 5
(3, 0) : 6
(3, 1) : 7
label column: survival_months
label dictionary: {(0, 0): 0, (0, 1): 1, (1, 0): 2, (1, 1): 3, (2, 0): 4, (2, 1): 5, (3, 0): 6, (3, 1): 7}
number of classes: 8
slide-level counts:  
 7    112
5     48
4     30
6     30
2     30
3     11
0     30
1     27
Name: label, dtype: int64
Patient-LVL; Number of samples registered in class 0: 30
Slide-LVL; Number of samples registered in class 0: 30
Patient-LVL; Number of samples registered in class 1: 27
Slide-LVL; Number of samples registered in class 1: 27
Patient-LVL; Number of samples registered in class 2: 30
Slide-LVL; Number of samples registered in class 2: 30
Patient-LVL; Number of samples registered in class 3: 11
Slide-LVL; Number of samples registered in class 3: 11
Patient-LVL; Number of samples registered in class 4: 30
Slide-LVL; Number of samples registered in class 4: 30
Patient-LVL; Number of samples registered in class 5: 48
Slide-LVL; Numbe

## 3. Load Model

In [47]:
cfc=[0.1,0.1,0.1] 
model=generate_model(args=args).to(device)

<All keys matched successfully>

In [ ]:
from utils.utils import NLLSurvLoss
loss_fn = {
    "surv_loss": NLLSurvLoss(alpha=args.alpha_surv),
    "surv_loss_g": NLLSurvLoss(alpha=args.alpha_surv),
    "surv_loss_p": NLLSurvLoss(alpha=args.alpha_surv),
    "kl_loss":nn.KLDivLoss(reduction='batchmean'),
    }

In [ ]:
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=args.lr, weight_decay=args.reg)

## 4. Runing 1 epoch Experiment on Train Loader

In [ ]:
all_loss = np.zeros((len(train_loader)))
all_risk_scores = np.zeros((len(train_loader)))
all_risk_scores_cf = np.zeros((len(train_loader)))
all_risk_scores_gp = np.zeros((len(train_loader)))
all_risk_scores_g = np.zeros((len(train_loader)))
all_risk_scores_p = np.zeros((len(train_loader)))

all_censorships = np.zeros((len(train_loader)))
all_event_times = np.zeros((len(train_loader)))

slide_ids = train_loader.dataset.slide_data['slide_id']

z_gkps = []
z_gs = []

train_loss_surv, train_loss = 0., 0.


In [48]:
model.train()
patient_results = {}


for batch_idx, (data_WSI, data_omic, label, event_time, c) in enumerate(train_loader):

        data_WSI = data_WSI.cuda()
        data_omic1 = data_omic[0][0].type(torch.FloatTensor).to(device)
        data_omic2 = data_omic[0][1].type(torch.FloatTensor).to(device)
        data_omic3 = data_omic[0][2].type(torch.FloatTensor).to(device)
        data_omic4 = data_omic[0][3].type(torch.FloatTensor).to(device)
        data_omic5 = data_omic[0][4].type(torch.FloatTensor).to(device)
        data_omic6 = data_omic[0][5].type(torch.FloatTensor).to(device)
        label = label.type(torch.LongTensor).cuda()
        c = c.type(torch.FloatTensor).cuda()

        slide_id = slide_ids.iloc[batch_idx]
        # get model output
        output = model(cfc=cfc,x_path=data_WSI, x_omic1=data_omic1, x_omic2=data_omic2, x_omic3=data_omic3, x_omic4=data_omic4, x_omic5=data_omic5, x_omic6=data_omic6)
        # calculate nll loss
        loss1=loss_fn['surv_loss'](hazards=output['hazards'], S=output['S'], Y=label, c=c)
        loss2=loss_fn['surv_loss_g'](hazards=output['g_hazards'], S=output['g_S'], Y=label, c=c)
        if args.p_branch:
            loss3=loss_fn['surv_loss_p'](hazards=output['p_hazards'], S=output['p_S'], Y=label, c=c)
        else:
            loss3=0
        # calculate kl loss 
        logits_rubi=output['hazards']
        nde = output['z_nde']
        p_te = torch.nn.functional.softmax(logits_rubi, -1).clone().detach()
        p_nde = torch.nn.functional.softmax(nde, -1)
        kl_loss = - p_te*p_nde.log()    
        kl_loss = kl_loss.sum(1).mean() 
        # integrate losses
        loss =loss1+loss2+loss3+kl_loss
        loss_value = loss.item()
        # 
        z_gkps.append(output['z_gkp'])
        z_gs.append(output['z_g'])
        ###
        all_loss[batch_idx]=loss_value
        all_risk_scores_cf[batch_idx] = -torch.sum(output['cf_S'], dim=1).detach().cpu().numpy()
        # all_risk_scores_gp[batch_idx] = -torch.sum(output['gp_S'], dim=1).detach().cpu().numpy()
        all_risk_scores_gp[batch_idx] = -torch.sum(output['S'], dim=1).detach().cpu().numpy()
        all_risk_scores_g[batch_idx] = -torch.sum(output['g_S'], dim=1).detach().cpu().numpy()
        if args.p_branch:
            all_risk_scores_p[batch_idx] = -torch.sum(output['p_S'], dim=1).detach().cpu().numpy()
        risk = -torch.sum(output['S'], dim=1).detach().cpu().numpy()
        all_risk_scores[batch_idx] = risk
        all_censorships[batch_idx] = c.item()
        all_event_times[batch_idx] = event_time

        train_loss_surv += loss_value
        train_loss += loss_value 



        if (batch_idx + 1) % 100 == 0:
            train_batch_str = 'batch {}, loss: {:.4f}, label: {}, event_time: {:.4f}, risk: {:.4f}'.format(
                batch_idx, loss_value, label.item(), float(event_time), float(risk))
            with open(os.path.join(args.writer_dir, 'log.txt'), 'a') as f:
                f.write(train_batch_str+'\n')
            f.close()
            print(train_batch_str)
        loss = loss / args.gc 
        loss.backward()

        if (batch_idx + 1) % args.gc == 0: 
            optimizer.step()
            optimizer.zero_grad()

# calculate loss and error for epoch
train_loss_surv /= len(train_loader)
train_loss /= len(train_loader)
c_index_train = concordance_index_censored((1-all_censorships).astype(bool), all_event_times, all_risk_scores, tied_tol=1e-08)[0]
# for update 
c_index_train_gp = concordance_index_censored((1-all_censorships).astype(bool), all_event_times, all_risk_scores_gp, tied_tol=1e-08)[0]
c_index_train_cf = concordance_index_censored((1-all_censorships).astype(bool), all_event_times, all_risk_scores_cf, tied_tol=1e-08)[0]
c_index_train_g = concordance_index_censored((1-all_censorships).astype(bool), all_event_times, all_risk_scores_g, tied_tol=1e-08)[0]
if args.p_branch:
    c_index_train_p = concordance_index_censored((1-all_censorships).astype(bool), all_event_times, all_risk_scores_p, tied_tol=1e-08)[0]

train_epoch_str = 'Epoch: {}, train_loss_surv: {:.4f}, train_loss: {:.4f}, train_c_index: {:.4f}'.format(
    0, train_loss_surv, train_loss, c_index_train)
print(train_epoch_str)

## 5. C-Index Result

In [ ]:
c_index = concordance_index_censored((1-all_censorships).astype(bool), all_event_times, all_risk_scores, tied_tol=1e-08)[0]
print("val c-index: {:.4f}".format(c_index))